# Source-Free Adaptation for flow-based NIDS — Phase 4 (revision experiments)
Answers reviewer comments #1 (one architecture), #4 (anchor contamination 20–50 %), #5 (label-budget, active-learning and one-class baselines), #7 (BARA with its fixable parts removed) and #10 (normalisation). The protocol and decision rules below are copied verbatim from the OSF pre-registration and must be registered **before** this notebook is run on real data.

**Block A — model families.** MLP 128-64 (reference), ResMLP (stem + 4 residual blocks, 9 BatchNorm layers), Random Forest, XGBoost. Per cell: source-only, oracle, EM prior correction (all models); BN-Adapt, TENT, SHOT, AaD and a target-label linear probe on the frozen representation (diagnostic only) for the two networks; source-only / BN-Adapt / TENT / EM with **target-statistics normalisation** and clipping rates for the MLP.

**Block B — alternatives to adaptation (MLP source model).** Label budgets k ∈ {10, 50, 100, 500}, random (at the target prior) and class-balanced sampling: head refit on frozen features, full fine-tune, MLP from scratch, RF from scratch. Uncertainty-based active learning (head refit) at 50/100/500 labels. Anchor of 1000 flows with contamination 0 / 5 / 20 / 50 %: Mahalanobis (pre-ReLU), Isolation Forest, autoencoder, One-Class SVM (input), SVDD (RBF one-class SVM on pre-ReLU features), trimmed PCA, aligned source head (plain covariance, no cleaning), BARA-fixed fusions, and the original BARA for reference.

**Primary analysis set:** 21 cells (12 directions × 2 priors minus the three cells whose target is BoT-IoT at its natural 99.6 % attack rate); 24 cells as sensitivity. Uncertainty: hierarchical crossed bootstrap (source and target datasets resampled independently, then seeds). Tests: Wilcoxon signed-rank, Holm-corrected within each hypothesis family.

In [1]:
# ===== 1. Setup =====
import os, glob, json, time, math, random, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import MinCovDet
from scipy.stats import wilcoxon
try:
    import xgboost as xgb; HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = '/content/drive/MyDrive'
except Exception:
    ROOT = '.'
OUT = f'{ROOT}/sfda_ids_phase4'; os.makedirs(OUT, exist_ok=True)
P1 = f'{ROOT}/sfda_ids_phase1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| xgboost:', HAS_XGB, '| results ->', OUT)

RUN = {'A': True, 'B': True}          # switch a block off to run the other one first; both resume from the CSVs
CFG = dict(
    datasets={'ton': 'dhoogla/nftoniotv2', 'unsw': 'dhoogla/nfunswnb15v2', 'bot': 'dhoogla/nfbotiotv2', 'cic': 'dhoogla/nfcsecicids2018v2'},
    required=['ton', 'unsw'], attack_rate={'ton': 0.6399, 'unsw': 0.0398, 'bot': 0.9964, 'cic': 0.1195},
    n_per_dataset=300_000, test_frac=0.3, seeds=[0, 1, 2], hidden=[128, 64], res_width=128, res_blocks=4,
    src_epochs=5, bs=512, lr=1e-3, priors=['natural', 0.05], n_target_train=100_000, n_target_test=40_000,
    artifacts=['MIN_TTL', 'MAX_TTL', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT'], clip=10.0,
    tent_lr=1e-3, shot_lr=1e-4, tent_epochs=1, shot_epochs=3, shot_beta=0.3, aad_epochs=3, aad_lr=1e-4, aad_k=5,
    rf_trees=200, rf_train_rows=100_000, xgb_trees=300, xgb_depth=8,
    budgets=[10, 50, 100, 500], al_start=10, al_checkpoints=[50, 100, 500], al_pool=20_000,
    ft_steps=200, ft_lr=1e-3, scratch_steps=300,
    anchor=1000, contaminations=[0.0, 0.05, 0.20, 0.50], alpha=0.01, ae_steps=1500, ridge=1e-3, mad_k=6.0, min_kept=100,
    ocsvm_nu=0.05, pca_var=0.95, pca_trim=0.2, pca_iters=2, probe_rows=20_000)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)
DROP = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'DNS_QUERY_ID', 'Dataset']
NEG = ['0', '0.0', 'benign', 'false', 'normal']
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Mounted at /content/drive
device: cuda | xgboost: True | results -> /content/drive/MyDrive/sfda_ids_phase4


In [2]:
# ===== 2. Data (reuses the Phase 1 samples on Drive; identical preprocessing, 34 features) =====
def label_of(df):
    cols = {c.lower(): c for c in df.columns}; lab = cols.get('label')
    return (~df[lab].astype(str).str.strip().str.lower().isin(NEG)).values

def load_nf(key, slug, n, seed=0):
    for path in (f'{P1}/data_{key}_{n}.parquet', f'{OUT}/data_{key}_{n}.parquet'):
        if os.path.exists(path): print(key, 'loaded', path); return pd.read_parquet(path)
    import kagglehub, pyarrow.parquet as pq
    root = kagglehub.dataset_download(slug)
    pfs = [pq.ParquetFile(f) for f in sorted(glob.glob(os.path.join(root, '**', '*.parquet'), recursive=True))]
    tot = sum(p.metadata.num_rows for p in pfs); r = CFG['attack_rate'][key]; rs = np.random.RandomState(seed); parts = []
    for p in pfs:
        for b in p.iter_batches(batch_size=1_000_000):
            d = b.to_pandas(); y = label_of(d)
            parts.append(d[rs.rand(len(d)) < np.where(y, min(1, .65 * n / (tot * r)), min(1, .65 * n / (tot * (1 - r))))])
    df = pd.concat(parts, ignore_index=True)
    if len(df) > n: df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    df.to_parquet(f'{OUT}/data_{key}_{n}.parquet'); return df

def prepare(df):
    cols = {c.lower(): c for c in df.columns}; lab, att = cols.get('label'), cols.get('attack')
    y = label_of(df).astype(np.int64)
    X = df.drop(columns=[c for c in [lab, att] + DROP if c is not None and c in df.columns])
    X = X.apply(lambda c: pd.to_numeric(c.astype(str) if str(c.dtype) in ('category', 'object', 'bool') else c, errors='coerce'))
    return X.loc[:, X.notna().any()].fillna(0.0), y

def build_data(raw):
    prep = {k: prepare(v) for k, v in raw.items()}
    common = [c for c in sorted(set.intersection(*[set(p[0].columns) for p in prep.values()])) if c not in CFG['artifacts']]
    data = {}
    for k, (X, y) in prep.items():
        V = X[common].values; Xl = (np.sign(V) * np.log1p(np.abs(V))).astype(np.float32)
        idx = np.random.RandomState(0).permutation(len(y)); nt = int(CFG['test_frac'] * len(y))
        data[k] = dict(Xtr=Xl[idx[nt:]], ytr=y[idx[nt:]], Xte=Xl[idx[:nt]], yte=y[idx[:nt]])
    print(f'{len(common)} features (artifacts removed)'); return data, common

In [3]:
# ===== 3. Models and SFDA methods =====
class Net(nn.Module):                     # reference MLP 128-64 (Phases 1-3)
    def __init__(s, d, h=None, k=2):
        super().__init__(); h = h or CFG['hidden']
        s.f = nn.Sequential(nn.Linear(d, h[0]), nn.BatchNorm1d(h[0]), nn.ReLU(), nn.Linear(h[0], h[1]), nn.BatchNorm1d(h[1]), nn.ReLU())
        s.h = nn.Linear(h[1], k)
    def forward(s, x): return s.h(s.f(x))

class Block(nn.Module):                   # residual block without the outer ReLU (added between blocks so f[:-1] is pre-ReLU)
    def __init__(s, w):
        super().__init__(); s.g = nn.Sequential(nn.Linear(w, w), nn.BatchNorm1d(w), nn.ReLU(), nn.Linear(w, w), nn.BatchNorm1d(w))
    def forward(s, x): return x + s.g(x)

class ResMLP(nn.Module):                  # stem + 4 residual blocks = 9 BatchNorm layers
    def __init__(s, d, k=2):
        super().__init__(); w = CFG['res_width']; layers = [nn.Linear(d, w), nn.BatchNorm1d(w), nn.ReLU()]
        for _ in range(CFG['res_blocks']): layers += [Block(w), nn.ReLU()]
        s.f = nn.Sequential(*layers); s.h = nn.Linear(w, k)
    def forward(s, x): return s.h(s.f(x))

ARCH = {'mlp': Net, 'resmlp': ResMLP}
def T(x): return torch.as_tensor(x, device=device)
def batches(n, bs, gen=None):
    perm = (torch.randperm(n, generator=gen) if gen is not None else torch.randperm(n)).to(device)
    return [perm[i:i + bs] for i in range(0, n, bs) if len(perm[i:i + bs]) > 1]

def train_sup(X, y, seed, epochs, arch='mlp', class_weight=False):
    set_seed(seed); m = ARCH[arch](X.shape[1]).to(device); opt = torch.optim.Adam(m.parameters(), CFG['lr'])
    Xt, yt = T(X), T(y); g = torch.Generator().manual_seed(seed); w = cw(y) if class_weight else None
    for _ in range(epochs):
        m.train()
        for b in batches(len(Xt), CFG['bs'], g):
            opt.zero_grad(); F.cross_entropy(m(Xt[b]), yt[b], weight=w).backward(); opt.step()
    return m.eval()

def cw(y):
    n1 = max(int(np.sum(y == 1)), 1); n0 = max(int(np.sum(y == 0)), 1)
    return torch.tensor([len(y) / (2 * n0), len(y) / (2 * n1)], dtype=torch.float32, device=device)

def train_steps(m, X, y, seed, steps, lr, bn_train):
    # small-label training: class-weighted CE, fixed number of steps; bn_train=False keeps BatchNorm statistics frozen
    set_seed(seed); opt = torch.optim.Adam(m.parameters(), lr); Xt, yt = T(X), T(y); w = cw(y); bs = min(len(y), 128)
    for _ in range(steps):
        m.train() if bn_train else m.eval()
        b = torch.randint(0, len(y), (bs,), device=device)
        if bn_train and bs < 2: break
        opt.zero_grad(); F.cross_entropy(m(Xt[b]), yt[b], weight=w).backward(); opt.step()
    return m.eval()

@torch.no_grad()
def proba(m, X):
    m.eval(); Xt = T(X)
    return torch.cat([F.softmax(m(Xt[i:i + 8192]), 1) for i in range(0, len(Xt), 8192)]).cpu().numpy()

@torch.no_grad()
def feats(m, X, pre=False):
    m.eval(); Xt = T(X); body = m.f[:-1] if pre else m.f
    return torch.cat([body(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)]).cpu().numpy().astype(np.float64)

def bn_adapt(m, X):
    m = copy.deepcopy(m)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d): mod.reset_running_stats(); mod.momentum = None
    m.train(); Xt = T(X)
    with torch.no_grad():
        for i in range(0, len(Xt), 1024):
            if len(Xt[i:i + 1024]) > 1: m(Xt[i:i + 1024])
    return m.eval()

def tent(m, X, seed, lr):
    set_seed(seed); m = copy.deepcopy(m); m.requires_grad_(False); params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d): mod.requires_grad_(True); params += [mod.weight, mod.bias]
    opt = torch.optim.Adam(params, lr); Xt = T(X)
    for _ in range(CFG['tent_epochs']):
        m.train()
        for b in batches(len(Xt), 256):
            lp = F.log_softmax(m(Xt[b]), 1); loss = -(lp.exp() * lp).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def shot(m, X, seed, lr):
    set_seed(seed); m = copy.deepcopy(m); m.h.requires_grad_(False)
    opt = torch.optim.Adam(m.f.parameters(), lr); Xt = T(X)
    for _ in range(CFG['shot_epochs']):
        m.eval()
        with torch.no_grad():
            fe = torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)])
            p = F.softmax(m.h(fe), 1); fn = F.normalize(fe, dim=1)
            c = F.normalize((p.T @ fn) / (p.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
            oh = F.one_hot(pl, 2).float(); c = F.normalize((oh.T @ fn) / (oh.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
        m.train()
        for b in batches(len(Xt), 256):
            out = m(Xt[b]); q = F.softmax(out, 1); ent = -(q * torch.log(q + 1e-5)).sum(1).mean()
            qb = q.mean(0); div = (qb * torch.log(qb + 1e-5)).sum()
            loss = ent + div + CFG['shot_beta'] * F.cross_entropy(out, pl[b]); opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def aad(m, X, seed):
    set_seed(seed); m = copy.deepcopy(m); m.h.requires_grad_(False)
    opt = torch.optim.Adam(m.f.parameters(), CFG['aad_lr']); Xt = T(X); K = CFG['aad_k']
    m.eval()
    with torch.no_grad():
        fe = torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)])
        FB = F.normalize(fe, dim=1); PB = F.softmax(m.h(fe), 1)
    total = CFG['aad_epochs'] * math.ceil(len(Xt) / 256); it = 0
    for _ in range(CFG['aad_epochs']):
        m.train()
        for b in batches(len(Xt), 256):
            feat = m.f(Xt[b]); q = F.softmax(m.h(feat), 1)
            with torch.no_grad():
                fn = F.normalize(feat, dim=1); FB[b] = fn; PB[b] = q.detach()
                sim = fn @ FB.T; sim[torch.arange(len(b), device=device), b] = -2
                Pn = PB[sim.topk(K, dim=1).indices]
            attract = -(q[:, None, :] * Pn).sum(-1).sum(1).mean()
            dot = q @ q.T; mask = 1 - torch.eye(len(b), device=device)
            disperse = (dot * mask).sum(1).mean() / (len(b) - 1) * K
            alpha = (1 + 10 * it / total) ** (-5.0); it += 1
            loss = attract + alpha * disperse; opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def em_prior(P_unlab, P_eval, pi_s, iters=200):
    pi_t = pi_s.copy()
    for _ in range(iters):
        q = P_unlab * (pi_t / pi_s); q /= q.sum(1, keepdims=True); new = q.mean(0)
        if np.abs(new - pi_t).max() < 1e-6: pi_t = new; break
        pi_t = new
    q = P_eval * (pi_t / pi_s); q /= q.sum(1, keepdims=True)
    return q, float(pi_t[1])

# ---- tree models (monotone per-feature transforms do not change them, so they use the log features without scaling) ----
def fit_tree(kind, X, y, seed, n_rows=None):
    rs = np.random.RandomState(seed)
    if n_rows and len(y) > n_rows: i = rs.choice(len(y), n_rows, replace=False); X, y = X[i], y[i]
    if kind == 'rf':
        return RandomForestClassifier(n_estimators=CFG['rf_trees'], n_jobs=-1, random_state=seed).fit(X, y)
    kw = dict(n_estimators=CFG['xgb_trees'], max_depth=CFG['xgb_depth'], learning_rate=0.1, tree_method='hist', random_state=seed, n_jobs=-1)
    try: return xgb.XGBClassifier(device='cuda' if device == 'cuda' else 'cpu', **kw).fit(X, y)
    except Exception: return xgb.XGBClassifier(**kw).fit(X, y)

def tree_proba(m, X):
    p = m.predict_proba(X)
    if p.shape[1] == 1:                                          # trained on one class only
        c = int(m.classes_[0]); q = np.zeros((len(X), 2)); q[:, c] = 1; return q
    return p

In [4]:
# ===== 4. Anchor scorers, label-budget learners, metrics =====
def metrics(y, pred, score=None):
    pos, neg = y == 1, y == 0
    tpr = ((pred == 1) & pos).sum() / max(pos.sum(), 1); fpr = ((pred == 1) & neg).sum() / max(neg.sum(), 1)
    auc = roc_auc_score(y, score) if score is not None and 0 < y.mean() < 1 and np.isfinite(score).all() and np.ptp(score) > 0 else np.nan
    return dict(macro_f1=f1_score(y, pred, average='macro', zero_division=0), attack_recall=float(tpr), fpr=float(fpr), auroc=float(auc),
                pred_attack_rate=float(np.mean(pred)))

def head_np(m):
    return m.h.weight.detach().cpu().numpy().astype(np.float64), m.h.bias.detach().cpu().numpy().astype(np.float64)
def attack_prob(Z, W, b, relu=False):
    if relu: Z = np.maximum(Z, 0)
    l = Z @ W.T + b; return 1 / (1 + np.exp(-np.clip(l[:, 1] - l[:, 0], -50, 50)))
def msqrt(S, inv=False):
    w, V = np.linalg.eigh((S + S.T) / 2); w = np.clip(w, 1e-6, None); return (V * (w ** (-0.5 if inv else 0.5))) @ V.T
def gauss(Z, robust=False, seed=0):
    keep = Z.std(0) > 1e-8; mu, C = Z.mean(0), np.cov(Z, rowvar=False)
    if robust and len(Z) > 2 * keep.sum() + 2:
        try:
            mcd = MinCovDet(random_state=seed).fit(Z[:, keep]); mu = mu.copy(); mu[keep] = mcd.location_
            C = C.copy(); C[np.ix_(keep, keep)] = mcd.covariance_
        except Exception as e: print('  MCD failed, plain covariance used:', str(e)[:60])
    return mu, C + CFG['ridge'] * np.eye(len(mu))
def maha(Z, mu, C):
    D = Z - mu; return np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', D, np.linalg.pinv(C), D), 0))
def ecdf(ref, x):
    r = np.sort(ref); return np.searchsorted(r, x, side='right') / len(r)
def outliers(s, kind):
    s = np.asarray(s, float)
    t = np.log(s + 1e-12) if kind == 'dist' else np.log(np.clip(s, 1e-12, 1 - 1e-12) / (1 - np.clip(s, 1e-12, 1 - 1e-12)))
    med = np.median(t); mad = 1.4826 * np.median(np.abs(t - med)) + 1e-12; return t > med + CFG['mad_k'] * mad

def robust_scale(Za, *Zs):
    med = np.median(Za, 0); iqr = np.subtract(*np.percentile(Za, [75, 25], 0)); s = np.where(iqr > 1e-8, iqr, Za.std(0) + 1e-6)
    return [(Z - med) / s for Z in (Za,) + Zs]

def ocsvm_scores(Za, Ze):
    Za, Ze = robust_scale(Za, Ze)
    m = OneClassSVM(kernel='rbf', gamma='scale', nu=CFG['ocsvm_nu']).fit(Za)
    return -m.decision_function(Za), -m.decision_function(Ze)

def trimmed_pca_scores(Za, Ze):
    # PCA fitted on the anchor, refitted after trimming the largest-score fraction (robust-PCA surrogate; no MCD)
    Za, Ze = robust_scale(Za, Ze); keep = np.ones(len(Za), bool)
    for it in range(CFG['pca_iters'] + 1):
        mu = Za[keep].mean(0); U, S, Vt = np.linalg.svd(Za[keep] - mu, full_matrices=False)
        var = S ** 2 / max(keep.sum() - 1, 1); k = int(np.searchsorted(np.cumsum(var) / var.sum(), CFG['pca_var']) + 1)
        P, lam = Vt[:k], var[:k] + 1e-9
        def score(Z):
            D = Z - mu; proj = D @ P.T; res = ((D - proj @ P) ** 2).sum(1)
            return (proj ** 2 / lam).sum(1) + res / (np.median(((Za[keep] - mu) - ((Za[keep] - mu) @ P.T) @ P) ** 2).sum() / keep.sum() + 1e-9)
        sa = score(Za)
        if it < CFG['pca_iters']: keep = sa <= np.quantile(sa, 1 - CFG['pca_trim'])
    return score(Za), score(Ze)

def ae_scores(Xa, Xe, seed):
    set_seed(seed); d = Xa.shape[1]
    ae = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 8), nn.ReLU(), nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, d)).to(device)
    opt = torch.optim.Adam(ae.parameters(), 1e-3, weight_decay=1e-5); A = T(Xa)
    for _ in range(max(1, CFG['ae_steps'] // math.ceil(len(A) / 64))):
        perm = torch.randperm(len(A), device=device)
        for i in range(0, len(A), 64):
            b = perm[i:i + 64]; opt.zero_grad(); F.mse_loss(ae(A[b]), A[b]).backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        E = T(Xe); ee = torch.cat([((ae(E[i:i + 8192]) - E[i:i + 8192]) ** 2).mean(1) for i in range(0, len(E), 8192)]); ea = ((ae(A) - A) ** 2).mean(1)
    return ea.cpu().numpy(), ee.cpu().numpy()

def resample(X, y, prior, n, seed):
    rs = np.random.RandomState(seed); att, ben = np.where(y == 1)[0], np.where(y == 0)[0]
    n = int(min(n, len(att) / prior, len(ben) / (1 - prior))); na = int(round(prior * n))
    idx = rs.permutation(np.concatenate([rs.choice(att, na, replace=False), rs.choice(ben, n - na, replace=False)]))
    return X[idx], y[idx]

def make_anchor(X, y, size, contam, seed):
    rs = np.random.RandomState(seed); na = int(round(contam * size)); ben, att = np.where(y == 0)[0], np.where(y == 1)[0]
    idx = np.concatenate([rs.choice(ben, size - na, replace=False), rs.choice(att, na, replace=False) if na else np.array([], int)])
    return X[idx]

def label_sample(y, k, how, seed):
    rs = np.random.RandomState(seed)
    if how == 'random': return rs.choice(len(y), k, replace=False)
    a, b = np.where(y == 1)[0], np.where(y == 0)[0]; ka = min(k // 2, len(a))
    return np.concatenate([rs.choice(a, ka, replace=False), rs.choice(b, k - ka, replace=False)])

def logreg(Z, y, seed):
    return LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=seed).fit(Z, y)

In [5]:
# ===== 5. Block A: model families, normalisation =====
def block_A(S, Tg, seed, priors, src_cache):
    rows, diag = [], []
    mu, sd = S['Xtr'].mean(0), S['Xtr'].std(0) + 1e-6
    def norm_with(X, m_, s_): return np.clip((X - m_) / s_, -CFG['clip'], CFG['clip']).astype(np.float32)
    def clip_rate(X, m_, s_): return float((np.abs((X - m_) / s_) >= CFG['clip']).mean())
    norm = lambda X: norm_with(X, mu, sd)
    pi_s = np.array([1 - S['ytr'].mean(), S['ytr'].mean()])
    nets = src_cache.setdefault('nets', {})                          # source models depend only on (source, seed)
    for a in ARCH:
        if a not in nets: nets[a] = train_sup(norm(S['Xtr']), S['ytr'], seed, CFG['src_epochs'], a)
    if 'trees' not in src_cache:
        src_cache['trees'] = {k: fit_tree(k, S['Xtr'], S['ytr'], seed, CFG['rf_train_rows']) for k in (['rf', 'xgb'] if HAS_XGB else ['rf'])}
    for prior in priors:
        pe = CFG['attack_rate'][Tg['key']] if prior == 'natural' else prior
        Xa_raw, ya = resample(Tg['Xtr'], Tg['ytr'], pe, CFG['n_target_train'], seed)
        Xb_raw, yb = resample(Tg['Xte'], Tg['yte'], pe, CFG['n_target_test'], seed + 100)
        R = []
        def add(model, method, P, extra=None):
            R.append(dict(model=model, method=method, **metrics(yb, P.argmax(1), P[:, 1]), **(extra or {})))
        # ---- neural models
        for a, src in src_cache['nets'].items():
            Xa, Xb = norm(Xa_raw), norm(Xb_raw)
            add(a, 'source_only', proba(src, Xb))
            add(a, 'bn_adapt', proba(bn_adapt(src, Xa), Xb))
            add(a, 'tent', proba(tent(src, Xa, seed, CFG['tent_lr']), Xb))
            if a != 'mlp':                                          # SHOT and AaD on the MLP are already in Phase 1
                add(a, 'shot', proba(shot(src, Xa, seed, CFG['shot_lr']), Xb))
                add(a, 'aad', proba(aad(src, Xa, seed), Xb))
            q, est = em_prior(proba(src, Xa), proba(src, Xb), pi_s); add(a, 'em_prior', q, dict(est_prior=est))
            add(a, 'oracle', proba(train_sup(Xa, ya, seed, CFG['src_epochs'], a), Xb))
            # diagnostic (uses target labels): linear probe on the frozen source representation
            n = min(CFG['probe_rows'], len(ya)); lr_ = logreg(feats(src, Xa[:n]), ya[:n], seed)
            add(a, 'probe_frozen_repr', lr_.predict_proba(feats(src, Xb)))
            if a == 'mlp':                                          # target-statistics normalisation (unlabeled target mean/std)
                mt, st = Xa_raw.mean(0), Xa_raw.std(0) + 1e-6; Xa_t, Xb_t = norm_with(Xa_raw, mt, st), norm_with(Xb_raw, mt, st)
                add(a, 'source_only_tnorm', proba(src, Xb_t)); add(a, 'bn_adapt_tnorm', proba(bn_adapt(src, Xa_t), Xb_t))
                add(a, 'tent_tnorm', proba(tent(src, Xa_t, seed, CFG['tent_lr']), Xb_t))
                q, est = em_prior(proba(src, Xa_t), proba(src, Xb_t), pi_s); add(a, 'em_prior_tnorm', q, dict(est_prior=est))
                diag += [dict(key='clip_rate_source_test', value=clip_rate(S['Xte'], mu, sd)),
                         dict(key='clip_rate_target_srcnorm', value=clip_rate(Xb_raw, mu, sd)),
                         dict(key='clip_rate_target_tgtnorm', value=clip_rate(Xb_raw, mt, st)),
                         dict(key='features_with_clipping_srcnorm', value=float((np.abs((Xb_raw - mu) / sd) >= CFG['clip']).any(0).sum()))]
                for d in diag[-4:]: d['prior'] = str(prior)
        # ---- tree models
        for k, src in src_cache['trees'].items():
            Pb = tree_proba(src, Xb_raw); add(k, 'source_only', Pb)
            q, est = em_prior(tree_proba(src, Xa_raw), Pb, pi_s); add(k, 'em_prior', q, dict(est_prior=est))
            add(k, 'oracle', tree_proba(fit_tree(k, Xa_raw, ya, seed, CFG['rf_train_rows']), Xb_raw))
        for r in R: r.update(prior=str(prior), true_attack_rate=float(yb.mean()))
        rows += R
    return rows, diag

In [6]:
# ===== 6. Block B: label budgets, active learning, anchors (MLP source model) =====
def block_B(S, Tg, seed, priors, src_cache):
    rows = []
    mu, sd = S['Xtr'].mean(0), S['Xtr'].std(0) + 1e-6
    norm = lambda X: np.clip((X - mu) / sd, -CFG['clip'], CFG['clip']).astype(np.float32)
    nets = src_cache.setdefault('nets', {})
    if 'mlp' not in nets: nets['mlp'] = train_sup(norm(S['Xtr']), S['ytr'], seed, CFG['src_epochs'], 'mlp')
    src = src_cache['nets']['mlp']; W, b = head_np(src)
    Zs = feats(src, norm(S['Xtr'][S['ytr'] == 0][:50_000]), pre=True)
    mu_s, C_s = Zs.mean(0), np.cov(Zs, rowvar=False) + CFG['ridge'] * np.eye(Zs.shape[1])
    for prior in priors:
        pe = CFG['attack_rate'][Tg['key']] if prior == 'natural' else prior
        Xa_raw, ya = resample(Tg['Xtr'], Tg['ytr'], pe, CFG['n_target_train'], seed)
        Xb_raw, yb = resample(Tg['Xte'], Tg['yte'], pe, CFG['n_target_test'], seed + 100)
        Xa, Xb = norm(Xa_raw), norm(Xb_raw); Fa, Fb = feats(src, Xa), feats(src, Xb)
        R = []
        def add(method, pred, score, **kw): R.append(dict(method=method, **metrics(yb, pred, score), **kw))
        p_src = proba(src, Xb)[:, 1]; add('source_only', (p_src > .5).astype(int), p_src)
        # ---- label budgets
        for k in CFG['budgets']:
            for how in ['random', 'balanced']:
                idx = label_sample(ya, k, how, seed + 7 * k); yk = ya[idx]; n_att = int(yk.sum()); one = n_att in (0, k)
                tag = dict(budget=k, sampling=how, n_attacks_labelled=n_att, single_class=int(one))
                if one:                                            # cannot fit a classifier; operator keeps the source model
                    for mth in ['headrefit', 'finetune', 'scratch_mlp', 'scratch_rf']: add(mth, (p_src > .5).astype(int), p_src, **tag)
                    continue
                h = logreg(Fa[idx], yk, seed); s = h.predict_proba(Fb)[:, 1]; add('headrefit', (s > .5).astype(int), s, **tag)
                m = train_steps(copy.deepcopy(src), Xa[idx], yk, seed, CFG['ft_steps'], CFG['ft_lr'] * 0.1, bn_train=False)
                s = proba(m, Xb)[:, 1]; add('finetune', (s > .5).astype(int), s, **tag)
                set_seed(seed); m = train_steps(Net(Xa.shape[1]).to(device), Xa[idx], yk, seed, CFG['scratch_steps'], CFG['ft_lr'], bn_train=True)
                s = proba(m, Xb)[:, 1]; add('scratch_mlp', (s > .5).astype(int), s, **tag)
                rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=seed, n_jobs=-1).fit(Xa_raw[idx], yk)
                s = tree_proba(rf, Xb_raw)[:, 1]; add('scratch_rf', (s > .5).astype(int), s, **tag)
        # ---- uncertainty-based active learning (head refit on frozen features)
        rs = np.random.RandomState(seed + 11); pool = rs.choice(len(ya), min(CFG['al_pool'], len(ya)), replace=False)
        lab = list(rs.choice(pool, CFG['al_start'], replace=False)); h = None
        while len(lab) < max(CFG['al_checkpoints']):
            yl = ya[lab]
            h = logreg(Fa[lab], yl, seed) if 0 < yl.sum() < len(yl) else None
            sp = h.predict_proba(Fa[pool])[:, 1] if h is not None else proba(src, Xa[pool])[:, 1]
            step = 10 if len(lab) < 100 else 50
            labset = set(lab); cand = [i for i in pool[np.argsort(np.abs(sp - 0.5))] if i not in labset][:step]; lab += cand
            if len(lab) in CFG['al_checkpoints']:
                yl = ya[lab]; tag = dict(budget=len(lab), sampling='active', n_attacks_labelled=int(yl.sum()), single_class=int(yl.sum() in (0, len(yl))))
                if tag['single_class']: add('al_headrefit', (p_src > .5).astype(int), p_src, **tag)
                else:
                    s = logreg(Fa[lab], yl, seed).predict_proba(Fb)[:, 1]; add('al_headrefit', (s > .5).astype(int), s, **tag)
        # ---- anchors
        for c in CFG['contaminations']:
            A = norm(make_anchor(Tg['Xtr'], Tg['ytr'], CFG['anchor'], c, seed)); tag = dict(contam=c)
            def addth(method, sa, se, keep=None):
                th = np.quantile(sa if keep is None else sa[keep], 1 - CFG['alpha']); add(method, (se > th).astype(int), se, **tag)
            Pa, Pe = feats(src, A, pre=True), feats(src, Xb, pre=True)
            mu_p, C_p = gauss(Pa); da, de = maha(Pa, mu_p, C_p), maha(Pe, mu_p, C_p); addth('maha_pre_plain', da, de)
            iso = IsolationForest(n_estimators=200, random_state=seed).fit(A); addth('iforest', -iso.score_samples(A), -iso.score_samples(Xb))
            ea, ee = ae_scores(A, Xb, seed); addth('autoencoder', ea, ee)
            sa, se = ocsvm_scores(A, Xb); addth('ocsvm_input', sa, se)
            va, ve = ocsvm_scores(Pa, Pe); addth('svdd_pre', va, ve)
            ta, te = trimmed_pca_scores(Pa, Pe); addth('trimmed_pca_pre', ta, te)
            Ap = msqrt(C_p, inv=True) @ msqrt(C_s)
            pa, pe_ = attack_prob((Pa - mu_p) @ Ap + mu_s, W, b, relu=True), attack_prob((Pe - mu_p) @ Ap + mu_s, W, b, relu=True)
            add('align_plain_head', (pe_ > .5).astype(int), pe_, **tag)
            fa, fe = (ecdf(da, da) + ecdf(pa, pa)) / 2, (ecdf(da, de) + ecdf(pa, pe_)) / 2; addth('bara_fixed_fused', fa, fe)
            fa, fe = (ecdf(va, va) + ecdf(pa, pa)) / 2, (ecdf(va, ve) + ecdf(pa, pe_)) / 2; addth('bara_svdd_fused', fa, fe)
            # original BARA (Phase 3 definition) for reference
            mu_r, C_r = gauss(Pa, True, seed); dra, dre = maha(Pa, mu_r, C_r), maha(Pe, mu_r, C_r); Ar = msqrt(C_r, inv=True) @ msqrt(C_s)
            qa, qe = attack_prob((Pa - mu_r) @ Ar + mu_s, W, b, relu=True), attack_prob((Pe - mu_r) @ Ar + mu_s, W, b, relu=True)
            keep = ~(outliers(dra, 'dist') | outliers(qa, 'prob') | (qa > 0.5))
            if keep.sum() >= CFG['min_kept']:
                mu_r, C_r = gauss(Pa[keep], True, seed); Ar = msqrt(C_r, inv=True) @ msqrt(C_s)
                dra, dre = maha(Pa, mu_r, C_r), maha(Pe, mu_r, C_r)
                qa, qe = attack_prob((Pa - mu_r) @ Ar + mu_s, W, b, relu=True), attack_prob((Pe - mu_r) @ Ar + mu_s, W, b, relu=True)
                keep = keep & ~(qa > 0.5)
            if keep.sum() < CFG['min_kept']: keep = np.ones(len(Pa), bool)
            fa = (ecdf(dra[keep], dra) + ecdf(qa[keep], qa)) / 2; fe = (ecdf(dra[keep], dre) + ecdf(qa[keep], qe)) / 2
            addth('bara_fused', fa, fe, keep)
        orc = train_sup(Xa, ya, seed, CFG['src_epochs'], 'mlp'); s = proba(orc, Xb)[:, 1]; add('oracle', (s > .5).astype(int), s)
        for r in R: r.update(prior=str(prior), true_attack_rate=float(yb.mean()))
        rows += R
    return rows

In [7]:
# ===== 7. Resumable runner =====
def run_all(data, pairs, prefix, seeds, blocks):
    files = {k: f'{prefix}_{k}_metrics.csv' for k in 'AB'}; dfile = f'{prefix}_A_diag.csv'
    done = {k: set() for k in 'AB'}
    for k, f in files.items():
        if os.path.exists(f): d = pd.read_csv(f); done[k] = set(zip(d.seed, d.src, d.tgt, d.prior.astype(str)))
    for seed in seeds:
        for s in sorted({p[0] for p in pairs}):
            cache = {}                                              # source models for (s, seed), reused across targets
            for t in [p[1] for p in pairs if p[0] == s]:
                for blk in blocks:
                    todo = [p for p in CFG['priors'] if (seed, s, t, str(p)) not in done[blk]]
                    if not todo: continue
                    t0 = time.time(); Tg = dict(data[t], key=t)
                    if blk == 'A':
                        rows, diag = block_A(data[s], Tg, seed, todo, cache)
                        for d in diag: d.update(seed=seed, src=s, tgt=t)
                        if diag: pd.DataFrame(diag).to_csv(dfile, mode='a', header=not os.path.exists(dfile), index=False)
                    else:
                        rows = block_B(data[s], Tg, seed, todo, cache)
                    for r in rows: r.update(seed=seed, src=s, tgt=t)
                    pd.DataFrame(rows).to_csv(files[blk], mode='a', header=not os.path.exists(files[blk]), index=False)
                    print(f'[{blk}] seed {seed} {s}->{t} priors {todo}: {len(rows)} rows ({(time.time()-t0)/60:.1f} min)', flush=True)
    return {k: pd.read_csv(f) for k, f in files.items() if os.path.exists(f)}

## Self-test on synthetic data — must print SELF-TEST PASSED

In [8]:
# ===== 8. Self-test =====
def synth(n, shift, prior, seed):
    rs = np.random.RandomState(seed); y = (rs.rand(n) < prior).astype(np.int64)
    return (rs.randn(n, 8).astype(np.float32) + y[:, None] * 1.5 + shift).astype(np.float32), y
fake = {}
for i, (k, sh, pr) in enumerate([('A', 0.0, 0.6), ('B', 0.8, 0.3)]):
    X, y = synth(12000, sh, pr, i); fake[k] = dict(Xtr=X[3000:], ytr=y[3000:], Xte=X[:3000], yte=y[:3000])
_save = copy.deepcopy(CFG)
CFG.update(n_target_train=4000, n_target_test=1500, attack_rate={'A': 0.6, 'B': 0.3}, ae_steps=20, src_epochs=1, anchor=300, rf_trees=20, xgb_trees=20,
           aad_epochs=1, shot_epochs=1, ft_steps=10, scratch_steps=10, al_pool=1000, probe_rows=2000, rf_train_rows=4000)
for f in glob.glob(f'{OUT}/selftest_*'): os.remove(f)
res = run_all(fake, [('A', 'B')], f'{OUT}/selftest', [0], ['A', 'B'])
CFG.clear(); CFG.update(_save)
a, bb = res['A'], res['B']
n_tree = 2 if HAS_XGB else 1
assert len(a) == 2 * (10 + 8 + 3 * n_tree), f'block A rows: {len(a)}'
assert len(bb) == 2 * (1 + 4 * 2 * 4 + 3 + 4 * 10 + 1), f'block B rows: {len(bb)}'
assert a.macro_f1.notna().all() and bb.macro_f1.notna().all(), 'NaN macro-F1'
assert os.path.exists(f'{OUT}/selftest_A_diag.csv')
res2 = run_all(fake, [('A', 'B')], f'{OUT}/selftest', [0], ['A', 'B'])        # resume: nothing new may be added
assert len(res2['A']) == len(a) and len(res2['B']) == len(bb), 'resume duplicated rows'
print('SELF-TEST PASSED')
print(a.pivot_table(index=['model', 'method'], columns='prior', values='macro_f1').round(3).to_string())

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [20:02:13] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[A] seed 0 A->B priors ['natural', 0.05]: 48 rows (0.2 min)
[B] seed 0 A->B priors ['natural', 0.05]: 154 rows (0.4 min)
SELF-TEST PASSED
prior                      0.05  natural
model  method                           
mlp    bn_adapt           0.399    0.855
       bn_adapt_tnorm     0.399    0.855
       em_prior           0.920    0.547
       em_prior_tnorm     0.722    0.974
       oracle             0.686    0.900
       probe_frozen_repr  0.950    0.972
       source_only        0.435    0.677
       source_only_tnorm  0.418    0.874
       tent               0.406    0.829
       tent_tnorm         0.404    0.864
resmlp aad                0.406    0.805
       bn_adapt           0.391    0.821
       em_prior           0.324    0.490
       oracle             0.925    0.968
       probe_frozen_repr  0.956    0.973
       shot               0.399    0.762
       source_only        0.342    0.576
       tent               0.381    0.797
rf     em_prior           0.326    0.431
 

## Real data

In [9]:
# ===== 9. Load and run =====
raw = {}
for k, slug in CFG['datasets'].items():
    try: raw[k] = load_nf(k, slug, CFG['n_per_dataset'])
    except Exception as e:
        if k in CFG['required']: raise
        print('SKIPPED', k, e)
DATA, FEATS = build_data(raw); del raw
keys = list(DATA); PAIRS = [(s, t) for s in keys for t in keys if s != t]
RES = run_all(DATA, PAIRS, f'{OUT}/real', CFG['seeds'], [k for k, v in RUN.items() if v])
print({k: len(v) for k, v in RES.items()})

ton loaded /content/drive/MyDrive/sfda_ids_phase1/data_ton_300000.parquet
unsw loaded /content/drive/MyDrive/sfda_ids_phase1/data_unsw_300000.parquet
bot loaded /content/drive/MyDrive/sfda_ids_phase1/data_bot_300000.parquet
cic loaded /content/drive/MyDrive/sfda_ids_phase1/data_cic_300000.parquet
34 features (artifacts removed)
[A] seed 2 ton->cic priors ['natural', 0.05]: 48 rows (2.2 min)


/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-743.730187589650200 > -743.788928456215899). You may want to try with a higher value of support_fraction (current value: 0.551).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-708.911518915017496 > -708.929499111217638). You may want to try with a higher value of support_fraction (current value: 0.551).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-860.121103062753718 > -860.150894844054847). You may want to try with a higher value of support_fraction (current value: 0.551).
  warnings.warn(
/usr/local

[B] seed 2 ton->cic priors ['natural', 0.05]: 154 rows (1.9 min)
[A] seed 2 unsw->ton priors ['natural', 0.05]: 48 rows (1.9 min)


/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1830.588819374321474 > -2174.864060854174113). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1830.588819374321474 > -2174.864060854174113). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1830.588819374321474 > -2174.864060854174113). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr

[B] seed 2 unsw->ton priors ['natural', 0.05]: 154 rows (1.7 min)
[A] seed 2 unsw->bot priors ['natural', 0.05]: 48 rows (1.1 min)


/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1868.548333469771933 > -1868.593279648524003). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1863.811710606382348 > -1863.896239166448822). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/covariance/_robust_covariance.py:185: RuntimeWarning: Determinant has increased; this should not happen: log(det) > log(previous_det) (-1783.508854094591698 > -1832.928206563776712). You may want to try with a higher value of support_fraction (current value: 0.535).
  warnings.warn(
/usr

[B] seed 2 unsw->bot priors ['natural', 0.05]: 154 rows (1.7 min)
[A] seed 2 unsw->cic priors ['natural', 0.05]: 48 rows (1.6 min)
[B] seed 2 unsw->cic priors ['natural', 0.05]: 154 rows (1.9 min)
{'A': 1728, 'B': 5544}


In [10]:
# ===== 10. Analysis and pre-registered decisions =====
CELL = ['src', 'tgt', 'prior']; DS = sorted(DATA) if 'DATA' in dir() else ['bot', 'cic', 'ton', 'unsw']; B = 2000
def primary(df): return df[~((df.tgt == 'bot') & (df.prior == 'natural'))]

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o): run = max(run, (m - k) * p[i]); adj[i] = min(1.0, run)
    return adj

def crossed_ci(per_seed, col, rng):
    # hierarchical crossed bootstrap: resample source datasets and target datasets independently, then seeds within each cell
    by_pair = {}
    for c, d in per_seed.groupby(CELL):
        v = d[[col, 'source_only', 'oracle']].dropna().values
        if len(v): by_pair.setdefault((c[0], c[1]), []).append(v)
    meds = []
    for _ in range(B):
        S, Tt = rng.choice(DS, len(DS)), rng.choice(DS, len(DS)); recs = []
        for s in S:
            for t in Tt:
                for v in by_pair.get((s, t), []):
                    m = v[rng.integers(0, len(v), len(v))].mean(0); recs.append((m[0] - m[1]) / (m[2] - m[1]))
        if recs: meds.append(np.median(recs))
    return float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5))

def summarise(df, methods, label, rng):
    df = primary(df.copy()); df['prior'] = df.prior.astype(str)
    W = df.pivot_table(index=CELL + ['seed'], columns='method', values='macro_f1').reset_index()
    C = W.groupby(CELL).mean(numeric_only=True); rec = C[methods].sub(C.source_only, axis=0).div(C.oracle - C.source_only, axis=0)
    p = [wilcoxon(rec[m].dropna()).pvalue if (rec[m].dropna() != 0).sum() > 5 else np.nan for m in methods]; ph = holm(np.nan_to_num(p, nan=1.0))
    out = []
    for j, m in enumerate(methods):
        lo, hi = crossed_ci(W, m, rng)
        out.append(dict(family=label, method=m, cells=int(rec[m].notna().sum()), median_recovery=float(rec[m].median()), ci_lo=lo, ci_hi=hi,
                        holm_p=float(ph[j]), cells_harmed=int((C[m] < C.source_only - 0.05).sum())))
    return pd.DataFrame(out), rec

rng = np.random.default_rng(2026); SUM, DEC = [], {}
A = pd.read_csv(f'{OUT}/real_A_metrics.csv'); A['prior'] = A.prior.astype(str)
Bm = pd.read_csv(f'{OUT}/real_B_metrics.csv'); Bm['prior'] = Bm.prior.astype(str)
# --- H1: SFDA on a deeper BN network; H1b: ranking failure of the source scorer for every model family
for model in A.model.unique():
    sub = A[A.model == model]; meths = [m for m in sub.method.unique() if m not in ('source_only', 'oracle')]
    s, _ = summarise(sub, meths, f'A:{model}', rng); SUM.append(s)
AU = primary(A[A.method == 'source_only']).groupby(['model'] + CELL).auroc.mean()
H1b = {m: dict(median_auroc=float(v.median()), cells_below_0_5=int((v < 0.5).sum()), cells=int(len(v))) for m, v in AU.groupby(level='model')}
s_res = SUM[[i for i, x in enumerate(SUM) if x.family.iloc[0] == 'A:resmlp'][0]] if 'resmlp' in A.model.unique() else None
sfda = ['bn_adapt', 'tent', 'shot', 'aad']
DEC['H1_sfda_fails_on_resmlp'] = bool(s_res is not None and all(not (r.median_recovery >= 0.2 and r.ci_lo > 0) for r in s_res[s_res.method.isin(sfda)].itertuples()))
DEC['H1b_ranking_failure_by_model'] = {m: bool(v['median_auroc'] < 0.60 and v['cells_below_0_5'] >= 7) for m, v in H1b.items()}
PR = primary(A[A.method == 'probe_frozen_repr']).groupby(['model'] + CELL).auroc.mean().groupby(level='model').median()
DEC['H2_representation_informative'] = {m: bool(v >= 0.95) for m, v in PR.items()}
# --- H6: normalisation
tn = A[(A.model == 'mlp')]; s_tn, _ = summarise(tn, ['source_only_tnorm', 'bn_adapt_tnorm', 'tent_tnorm', 'em_prior_tnorm', 'bn_adapt', 'tent', 'em_prior'], 'A:mlp_tnorm', rng); SUM.append(s_tn)
g = s_tn.set_index('method').median_recovery
DEC['H6_normalisation_not_the_cause'] = bool(max(abs(g['bn_adapt_tnorm'] - g['bn_adapt']), abs(g['tent_tnorm'] - g['tent'])) < 0.10 and g['source_only_tnorm'] < 0.2)
DG = pd.read_csv(f'{OUT}/real_A_diag.csv'); print('clipping diagnostics (median over runs):'); print(DG.groupby('key').value.describe()[['50%', 'max']].round(4))
# --- H3: label budgets and active learning
LB = Bm[Bm.method.isin(['headrefit', 'finetune', 'scratch_mlp', 'scratch_rf', 'al_headrefit'])].copy()
LB['method'] = LB.method + '@' + LB.sampling + '/' + LB.budget.astype(int).astype(str)
ref = Bm[Bm.method.isin(['source_only', 'oracle'])]
s_lb, rec_lb = summarise(pd.concat([LB, ref]), sorted(LB.method.unique()), 'B:labels', rng); SUM.append(s_lb)
# --- H4/H5: anchors under contamination, BARA-fixed
AN = Bm[Bm.contam.notna() & ~Bm.method.isin(['source_only', 'oracle'])].copy(); AN['method'] = AN.method + '@c' + AN.contam.astype(str)
s_an, _ = summarise(pd.concat([AN, ref]), sorted(AN.method.unique()), 'B:anchor', rng); SUM.append(s_an)
an = s_an.set_index('method').median_recovery
best_clean = max(an[[i for i in an.index if i.endswith('@c0.0')]])
rnd = s_lb[s_lb.method.str.startswith('headrefit@random/')].assign(k=lambda d: d.method.str.split('/').str[1].astype(int)).sort_values('k')
DEC['H3_smallest_random_budget_beating_best_clean_anchor'] = next((int(r.k) for r in rnd.itertuples() if r.median_recovery > best_clean), None)
DEC['H4_anchor_useless_at_20pct'] = bool(max(an[[i for i in an.index if i.endswith('@c0.2')]]) <= 0.10)
ah = primary(Bm[Bm.method == 'align_plain_head']).groupby(CELL + ['contam']).auroc.mean().xs(0.0, level='contam')
DEC['H5_alignment_hypothesis_refuted'] = bool(ah.median() < 0.5)
DEC['H5_bara_fixed_not_better_than_maha_at_5pct'] = bool(max(an['bara_fixed_fused@c0.05'], an['bara_svdd_fused@c0.05']) - an['maha_pre_plain@c0.05'] < 0.10)
S = pd.concat(SUM); S.to_csv(f'{OUT}/summary_primary21.csv', index=False)
pd.set_option('display.width', 250); print(S.round(3).to_string(index=False))
print('\nsource-score AUROC by model:', json.dumps(H1b, indent=1)); print('probe AUROC by model:', PR.round(3).to_dict())
json.dump(DEC, open(f'{OUT}/decision.json', 'w'), indent=2); print('\nDECISIONS', json.dumps(DEC, indent=2))

clipping diagnostics (median over runs):
                                   50%      max
key                                            
clip_rate_source_test           0.0003   0.0004
clip_rate_target_srcnorm        0.0017   0.0275
clip_rate_target_tgtnorm        0.0002   0.0005
features_with_clipping_srcnorm  9.0000  14.0000
     family                   method  cells  median_recovery  ci_lo  ci_hi  holm_p  cells_harmed
      A:mlp                 bn_adapt     21           -0.275 -0.374  0.090   0.044            15
      A:mlp                     tent     21           -0.239 -0.477  0.078   0.044            14
      A:mlp                 em_prior     21            0.001 -0.058  0.066   0.683             2
      A:mlp        probe_frozen_repr     21            0.889  0.691  0.971   0.000             0
      A:mlp        source_only_tnorm     21           -0.080 -0.235  0.039   0.140             9
      A:mlp           bn_adapt_tnorm     21           -0.216 -0.323  0.030   0.044       